# Worked solutions

This notebook is identical to the student version except that every 🔵 `# TODO` has been filled in, **with commentary on why the answer is what it is** rather than just the code. Read the comments — the reasoning is the point, not the syntax.

Everything else, including the ✏️ YOUR TURN cells, is unchanged: those have no single right answer.


# Module C — Blood and CSF biomarkers

## Can a blood test tell us who has Alzheimer's disease?

> ⚠️ **Simulated data — realistic numbers, invented people. Section 1 explains exactly why.**

### What you will be able to do by the end

1. read a clinical biomarker table and say what every column measures in the body
2. spot missing data that is *not* missing at random, and say why that matters
3. explain what an assay detection limit and a batch effect do to a model
4. train and compare five different models, and change their settings yourself
5. read a confusion matrix as a clinician would, and say what a missed diagnosis costs

### The data

**This module's table is simulated.** No open dataset of individual Aβ42, p-tau181 and NfL measurements exists that we are allowed to redistribute, so 420 participants were drawn from published cohort summaries: the group means, the spread, the assay batch offsets and the detection limit are all set to realistic published values. The biology is real; the people are not. Every other module you might pick today uses real measured data.

### How to work through this notebook

Run the cells in order, top to bottom. The notebook is split into four sections:

| | Section | What happens |
|---|---|---|
| 1 | **Understand the data** | Meet every column and every person in the table |
| 2 | **Quality control** | Find the flaws before they fool you |
| 3 | **Build models** | Start from something trivial, then climb |
| 4 | **Read the results** | Turn numbers into a clinical judgement |

Look out for these markers:

- ✏️ **YOUR TURN** — change the value shown, re-run the cell, watch the figure change. Everyone does these.
- 🟢 run and read · 🔵 write a little code · ⚫ take home
- 🧠 a question to think about; the answer is hidden underneath, so try first

**In a hurry?** Skim section 2 (run the cells, read the figures, skip the ✏️ turns) and spend your time on sections 3 and 4.

---

*Teaching material. Nothing here is a diagnostic tool, and no result in this notebook is clinical evidence.*


In [ ]:
# Run me first. This finds the project folder, loads the shared helpers,
# and prints exactly where this module's data came from.
from pathlib import Path
import sys
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plots
from data import load_data, load_extra, provenance
from models import split_data, train_model, evaluate, compare_models, sweep_parameter, MODEL_CHOICES

pd.set_option('display.width', 160)
print(provenance('C'))


---
# 1 · Understand the data

Before any modelling, you need to be able to say out loud what a single row is. Here, one row is **one person, at one memory-clinic visit**.


### 1.1 Load the table and look at it

`load_data('C')` returns a pandas *DataFrame* — a table, like a spreadsheet, where every column has a name and a type. `.shape` gives (rows, columns) and `.head()` shows the first few rows.


In [ ]:
df = load_data('C')
print('This table has', df.shape[0], 'people and', df.shape[1], 'columns.\n')
df.head()


### 1.2 What does each column actually measure?

This is the part biologists find obvious and computer scientists do not, and vice versa. Read it — you cannot judge a model without it.

| Column | What it is | Why it matters in Alzheimer's disease |
|---|---|---|
| `age` | years | The single strongest risk factor for AD. It will haunt every model you build today. |
| `sex` | F / M | Around two-thirds of people with AD are women; some of that is longer life expectancy, some is not. |
| `education_years` | years of schooling | A proxy for **cognitive reserve**: more education tends to delay the *appearance* of symptoms, not the underlying pathology. |
| `apoe4_carrier` | 0 / 1 | Carrying at least one *APOE* ε4 allele, the biggest common genetic risk factor. |
| `site` | site_1/2/3 | Which clinic collected the sample. Different clinics used different assay lots — remember this. |
| `diagnosis` | CN / MCI / AD | **The label.** CN = cognitively normal. MCI = mild cognitive impairment, a fuzzy in-between state. AD = diagnosed Alzheimer's dementia. |
| `ab42_pg_ml` | amyloid-β 42, pg/ml | Aβ42 is the sticky peptide that forms plaques. Counter-intuitively it goes **down** in blood/CSF in AD — it is being deposited in the brain instead of floating around. |
| `ab40_pg_ml` | amyloid-β 40, pg/ml | A related peptide that is *not* strongly deposited. Mostly reflects how much amyloid you make overall. |
| `ab42_40_ratio` | Aβ42 ÷ Aβ40 | The ratio cancels out person-to-person differences in total production. Keep an eye on whether it beats Aβ42 alone. |
| `ptau181_pg_ml` | phosphorylated tau 181, pg/ml | Tangle pathology. Goes **up** in AD, and is fairly specific to it. |
| `nfl_pg_ml` | neurofilament light, pg/ml | Leaks out of damaged axons. Goes up in AD — and in stroke, MS, ALS, ageing. Sensitive, not specific. |
| `gfap_pg_ml` | glial fibrillary acidic protein, pg/ml | Astrocyte activation; rises early in the amyloid cascade. |
| `mmse` | 0–30 | Mini-Mental State Examination, a bedside cognitive test. 30 is perfect, under ~24 suggests impairment. **Note where this number comes from — we come back to it.** |


### 1.3 Who is in this cohort?

**Predict before you run:** the three diagnostic groups — will they be equally sized? Should the AD group be older or younger than the CN group?


In [ ]:
plots.plot_class_balance(df['diagnosis'], title='Diagnostic groups in the cohort')
plt.show()

plots.plot_by_group(df, 'age', 'diagnosis', unit='(years)',
                    title='Age distribution by diagnosis — notice the overlap, and the shift')
plt.show()

print(df.groupby('diagnosis')[['age', 'education_years', 'apoe4_carrier']].mean().round(2))


### 1.4 ✏️ Your turn — look at one biomarker at a time

Change `BIOMARKER` below to any of these and re-run the cell:

`'ab42_pg_ml'` · `'ab40_pg_ml'` · `'ab42_40_ratio'` · `'ptau181_pg_ml'` · `'nfl_pg_ml'` · `'gfap_pg_ml'` · `'mmse'`

For each one, ask yourself: **do the three coloured histograms sit on top of each other, or apart?** A marker whose groups overlap completely cannot separate patients no matter how clever the model is.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Pick a biomarker. Try at least three of them.
#   Watch the 'separation score' printed underneath: it is how many
#   standard deviations apart the CN and AD group means are.
# ==========================================================================
BIOMARKER = 'ab42_40_ratio'

plots.plot_by_group(df, BIOMARKER, 'diagnosis',
                    title=f'{BIOMARKER} by diagnosis')
plt.show()

summary = df.groupby('diagnosis')[BIOMARKER].agg(['mean', 'std', 'count']).round(3)
print(summary)

cn, ad = df[df.diagnosis == 'CN'][BIOMARKER].dropna(), df[df.diagnosis == 'AD'][BIOMARKER].dropna()
pooled_sd = np.sqrt((cn.var() + ad.var()) / 2)
separation = abs(cn.mean() - ad.mean()) / pooled_sd
print(f'\nSeparation score (CN vs AD): {separation:.2f} standard deviations')
print('Under 0.5 = the groups are basically the same. Over 1.5 = a strong single marker.')


### 1.5 Wait, that's odd

Let's rank every numeric column by how strongly it tracks an AD diagnosis. One of them is going to look suspiciously good.


In [ ]:
is_ad = (df['diagnosis'] == 'AD').astype(int)
strength = df.select_dtypes('number').apply(lambda column: column.corr(is_ad)).drop(labels=[], errors='ignore')
strength = strength.dropna().sort_values(key=abs, ascending=False)

plots.plot_importance(strength.index, strength.values,
                      title='Correlation of each measurement with an AD diagnosis',
                      xlabel='correlation (negative = lower in AD)')
plt.show()
print(strength.round(3))


🧠 **Think first:** `mmse` is the strongest signal in the table by a wide margin. Should we be pleased?

<details>
<summary>Click for one good answer</summary>

No. The MMSE is a **cognitive test**, and a clinician used the patient's cognitive performance to assign the diagnosis in the first place. Predicting the diagnosis from the MMSE is close to predicting the label from the label. It is not *impossible* to use — an MMSE is cheap and a blood draw is not — but a model that leans on it is not telling you anything about **blood biomarkers**, which is the actual question this module asks. We come back to this in section 2.5.

</details>


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** re-run cell 1.4 with `'nfl_pg_ml'` and with `'ptau181_pg_ml'` and decide which you would rather have as a screening test.
- 🔵 **If you want to write code:** make a scatter plot of `ab42_40_ratio` against `ptau181_pg_ml`, coloured by diagnosis, using `plots.plot_scatter(df['ab42_40_ratio'], df['ptau181_pg_ml'], colour_by=df['diagnosis'], ...)`. Do the two markers carry the same information, or different information?
- ⚫ **Take home:** the real Aβ42/40 ratio is measured on several different analytical platforms that disagree with each other. Look up 'Aβ42/40 harmonisation' and consider what that means for a threshold published by one lab.


---
# 2 · Quality control

This is the section that matters most and gets skipped most. Four flaws live in this table, and each one can make a model look better than it is.


### 2.1 Flaw one — missing values that are not missing at random

Some p-tau181 and GFAP results are missing. The dangerous question is not *how many*, it is **who**. If sicker patients are more likely to miss a blood draw, then 'has a p-tau result' is itself a clue about diagnosis, and any analysis that quietly drops those rows is now studying a different, healthier population.


In [ ]:
plots.plot_missingness(df, title='Missing values, as a percentage of all 420 people')
plt.show()

# The real question: is a missing p-tau equally likely in every group?
missing_rate = df.assign(missing=df['ptau181_pg_ml'].isna()).groupby('diagnosis')['missing'].mean()
plots.plot_score_comparison(missing_rate.index.tolist(), (100 * missing_rate).tolist(),
                            title='Percentage of people with NO p-tau181 result, by diagnosis',
                            ylabel='percent missing')
plt.show()
print('If these bars are not level, the missingness carries information about the diagnosis.')


### 2.2 ✏️ Your turn — how you fill the gaps changes the answer

You have three sensible options, and they genuinely disagree:

- `'median'` — fill each gap with the middle value of that column. Safe, ignores the person.
- `'mean'` — fill with the average. Pulled around by extreme values.
- `'drop'` — throw away every row with any gap. Honest-looking, but see 2.1.

Change `HOW_TO_HANDLE_MISSING` and re-run. The figure shows how the held-out score moves, and how many people you have left.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Try all three: 'median', 'mean', 'drop'
#   Then answer: does the best score belong to the best method?
# ==========================================================================
HOW_TO_HANDLE_MISSING = 'median'

feature_columns = ['age', 'sex', 'education_years', 'apoe4_carrier',
                   'ab42_40_ratio', 'ptau181_pg_ml', 'nfl_pg_ml', 'gfap_pg_ml']

scores, labels, sizes = [], [], []
for strategy in ['median', 'mean', 'drop']:
    table = df.dropna(subset=feature_columns) if strategy == 'drop' else df
    X = table[feature_columns]
    y = (table['diagnosis'] == 'AD').astype(int)
    X_train, X_test, y_train, y_test = split_data(X, y)
    fill = 'median' if strategy == 'drop' else strategy
    model = train_model('logistic', X_train, y_train, impute=fill)
    scores.append(evaluate(model, X_test, y_test)['balanced_accuracy'])
    labels.append(f'{strategy}\n(n={len(table)})')
    sizes.append(len(table))

highlight = ['#e08214' if s.startswith(HOW_TO_HANDLE_MISSING) else '#2c6fbb' for s in ['median', 'mean', 'drop']]
plots.plot_score_comparison(labels, scores, colours=highlight, reference=0.5,
                            title=f'Your choice ({HOW_TO_HANDLE_MISSING}) is in orange')
plt.show()
print('Dropping rows costs you', len(df) - min(sizes), 'people — and not a random', len(df) - min(sizes), 'people.')


### 2.3 Flaw two — the assay cannot see below its detection limit

Every immunoassay has a floor. Below it the machine reports the floor, not the true value. In this dataset the p-tau181 assay bottoms out at **8.0 pg/ml**. Those people do not have a p-tau of exactly 8.0 — we simply do not know what they have. This is called **left censoring**.


In [ ]:
limit = 8.0
at_limit = (df['ptau181_pg_ml'] <= limit).sum()

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(df['ptau181_pg_ml'].dropna(), bins=40, color='#2c6fbb')
ax.axvline(limit, color='#c0392b', linewidth=2)
ax.annotate(f'detection limit\n{at_limit} people pile up here', (limit, ax.get_ylim()[1] * 0.7),
            xytext=(20, 0), textcoords='offset points', color='#c0392b', fontsize=9)
ax.set_xlabel('p-tau181 (pg/ml)'); ax.set_ylabel('number of people')
ax.set_title('The spike on the left is not biology, it is the machine')
plt.tight_layout(); plt.show()


### 2.4 Flaw three — the three clinics did not use the same assay lot

`site` records which clinic drew the blood. If one clinic's assay reads systematically high, then 'which clinic' becomes a fake biomarker. It only becomes a *disaster* if the clinics also recruited different kinds of patient — check both.


In [ ]:
plots.plot_group_means(df, 'ptau181_pg_ml', 'site', unit='(pg/ml)',
                       title='p-tau181 by collection site — same biology, different machines')
plt.show()

composition = pd.crosstab(df['site'], df['diagnosis'], normalize='index').round(3) * 100
print('Percentage of each site\'s patients in each diagnostic group:')
print(composition)
print('\nIf these rows look alike, site is only noise. If they differ, site is a confounder.')


### 2.5 Flaw four — leakage, in two flavours

**Leakage** is when information that would not be available at prediction time sneaks into training. It is the single most common reason a published medical AI result fails to replicate.

*Flavour one — preprocessing leakage.* If you scale or impute using the whole dataset and *then* split, the training set has secretly seen the test set's average.

*Flavour two — label leakage.* If a feature is part of how the label was decided (`mmse`, here), the model gets to peek at the answer.

We do the wrong thing on purpose, once, so you can see the size of the lie.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

columns_no_mmse = ['age', 'sex', 'education_years', 'apoe4_carrier',
                   'ab42_40_ratio', 'ptau181_pg_ml', 'nfl_pg_ml', 'gfap_pg_ml']
columns_with_mmse = columns_no_mmse + ['mmse']
y_all = (df['diagnosis'] == 'AD').astype(int)

# --- the WRONG way: impute and scale the whole table first, then split ---
numeric = df[columns_no_mmse].select_dtypes('number')
filled = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(numeric), columns=numeric.columns)
scaled = pd.DataFrame(StandardScaler().fit_transform(filled), columns=numeric.columns)
X_train, X_test, y_train, y_test = split_data(scaled, y_all)
leaky = LogisticRegression(max_iter=2000, class_weight='balanced').fit(X_train, y_train)
leaky_score = evaluate(leaky, X_test, y_test)['auroc']

# --- the RIGHT way: split first, fit the preprocessing inside the pipeline ---
X_train, X_test, y_train, y_test = split_data(df[columns_no_mmse], y_all)
clean_score = evaluate(train_model('logistic', X_train, y_train), X_test, y_test)['auroc']

# --- and the label-leakage version: let the model see the cognitive test ---
X_train_m, X_test_m, y_train_m, y_test_m = split_data(df[columns_with_mmse], y_all)
mmse_score = evaluate(train_model('logistic', X_train_m, y_train_m), X_test_m, y_test_m)['auroc']

plots.plot_score_comparison(
    ['scaled before splitting\n(leaky)', 'split first\n(honest)', 'honest, but shown\nthe MMSE'],
    [leaky_score, clean_score, mmse_score],
    colours=['#c0392b', '#2c6fbb', '#e08214'], reference=0.5,
    title='Three AUROCs. Only the blue one answers the question we asked.', ylabel='AUROC')
plt.show()


🧠 **Think first:** The MMSE model scores highest of all three. Why is it still the wrong model for this module's question?

<details>
<summary>Click for one good answer</summary>

Because the question was *"can a **blood test** identify Alzheimer's disease?"* — and that model's answer is mostly "a cognitive test can". We already knew that; it is how the diagnosis was made. A model is only as useful as the decision it would change, and this one changes nothing: any clinic that can run an MMSE already has the MMSE. The blood-only model, at a lower AUROC, is the one that would actually add information — for instance in a GP surgery, before a memory-clinic referral.

</details>


### 2.6 ✏️ Your turn — decide what goes into the model

You are now the analyst. Switch each of these on and off, re-run, and watch the score move. There is no single right answer — but you should be able to *justify* the one you pick.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Set each of these to True or False and re-run.
#   Suggested experiments:
#     (a) everything False except biomarkers -> the honest blood-only model
#     (b) USE_MMSE = True                    -> watch it jump, then ask why
#     (c) USE_SITE = True                    -> does knowing the clinic help? should it?
# ==========================================================================
USE_MMSE   = False   # the cognitive test the diagnosis was based on
USE_SITE   = False   # which clinic collected the sample
USE_AGE    = True    # age and sex
USE_RATIO  = True    # Abeta42/40 ratio instead of Abeta42 alone

chosen = ['ptau181_pg_ml', 'nfl_pg_ml', 'gfap_pg_ml', 'apoe4_carrier', 'education_years']
chosen += ['ab42_40_ratio'] if USE_RATIO else ['ab42_pg_ml']
if USE_AGE:
    chosen += ['age', 'sex']
if USE_SITE:
    chosen += ['site']
if USE_MMSE:
    chosen += ['mmse']

X_train, X_test, y_train, y_test = split_data(df[chosen], y_all)
chosen_model = train_model('logistic', X_train, y_train)
chosen_metrics = evaluate(chosen_model, X_test, y_test)

plots.plot_score_comparison(list(chosen_metrics), list(chosen_metrics.values()),
                            colours=['#2c6fbb'] * 5,
                            title=f'Your feature set: {len(chosen)} columns', ylabel='score')
plt.show()
print('Columns you gave the model:', ', '.join(chosen))


### 2.7 QC verdict — write it down before you model

**Usable? Yes, with three caveats.**

1. p-tau181 is missing more often in the sicker groups, so any row-dropping analysis studies a healthier cohort than the one you meant to study. Impute inside the pipeline instead.
2. p-tau181 is left-censored at 8 pg/ml. Values at the floor are upper bounds, not measurements. A model will happily treat that spike as a real cluster.
3. Site shifts the p-tau readings. Unless the sites recruited identical patients, some of any 'biomarker' signal is really 'which building'.

**And a decision:** we exclude `mmse` from the main model, because this module asks what a *blood test* can do. We will report the blood-only number as the headline.

*(**Express path:** you can start from section 3 — run its catch-up cell first and everything below stands alone.)*


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** go back to 2.6 and find the smallest set of columns that still reaches an AUROC above 0.85.
- 🔵 **If you want to write code:** write a cell that replaces the censored p-tau values (≤ 8.0) with `np.nan` and lets the imputer handle them as genuinely unknown. Does the model get better, worse, or just more honest?
- ⚫ **Take home:** read about **inverse probability weighting**, the standard statistical fix for data that is missing *because of* the thing you are studying.


---
# 3 · Build models

The rule is always the same: **start with something so simple it is almost insulting, then only add complexity that pays for itself.** If a random forest cannot beat a single blood test with a threshold on it, the random forest has earned nothing.


### 🚏 Taking the Express path? Run this one cell first

It rebuilds everything sections 3 and 4 need, so you can start here without having run sections 1 and 2 yourself. **If you did run them, run this anyway** — it just redefines the same things and costs a second.


In [ ]:
# Express catch-up: safe to run whether or not you did sections 1 and 2.
df = load_data('C')
y_all = (df['diagnosis'] == 'AD').astype(int)
print(f'{len(df)} participants; {y_all.sum()} with an AD diagnosis. Ready for section 3.')


### 3.1 Two baselines to beat

*Baseline zero:* always say "not AD". It gets most people right, because most people are not AD. That is why plain accuracy is a bad metric here.

*Baseline one:* pick a single biomarker and a cutoff. This is genuinely how biomarkers are used in clinics today. The ROC curve shows every possible cutoff at once.


In [ ]:
features = ['age', 'sex', 'education_years', 'apoe4_carrier',
            'ab42_40_ratio', 'ptau181_pg_ml', 'nfl_pg_ml', 'gfap_pg_ml']
X = df[features]
y = (df['diagnosis'] == 'AD').astype(int)
X_train, X_test, y_train, y_test = split_data(X, y)
print(f'{len(X_train)} people to learn from, {len(X_test)} held back to test on.')
print(f'{y_test.sum()} of the {len(y_test)} held-out people actually have AD.\n')

# Baseline zero
dumb = train_model('baseline', X_train, y_train)
print('Always saying "not AD":')
print(f"  plain accuracy      {(dumb.predict(X_test) == y_test).mean():.3f}   <- looks fine!")
print(f"  balanced accuracy   {evaluate(dumb, X_test, y_test)['balanced_accuracy']:.3f}   <- the honest version\n")

# Baseline one: a single marker, every possible cutoff
single = -X_test['ptau181_pg_ml'].fillna(X_train['ptau181_pg_ml'].median())
plots.plot_roc_pr(y_test, -single, title='One marker (p-tau181), every possible cutoff')
plt.show()


### 3.2 The model ladder

Now five models on the identical split. Each one is a different *shape* of decision rule:

| Model | The idea, in one sentence |
|---|---|
| `baseline` | Always guess the commonest answer. |
| `logistic` | Add up the markers with weights, squash into a probability. Straight-line boundaries, readable coefficients. |
| `knn` | Find the most similar patients we have seen and copy their diagnosis. |
| `random_forest` | Ask hundreds of slightly different flowcharts and take a vote. Handles interactions and curvature. |
| `svm` | Draw the boundary with the widest possible empty margin around it. |
| `mlp` | A small neural network. With 300 patients, do not expect miracles. |

**Predict before you run:** which will win? Write your guess down.


In [ ]:
ladder = ['baseline', 'logistic', 'knn', 'random_forest', 'svm', 'mlp']
comparison = compare_models(ladder, X_train, y_train, X_test, y_test)
display(comparison)

plots.plot_model_comparison(comparison, metric='auroc',
                            title='Blood-biomarker models on the same held-out patients (AUROC)')
plt.show()
print('The dashed line at 0.5 is a coin flip. Anything near it has learned nothing.')


### 3.3 ✏️ Your turn — turn the dial and watch it overfit

Every model has settings (*hyperparameters*). One of them usually controls **how much the model is allowed to contort itself around the training data**. Turn it too far and the model memorises the training patients instead of learning about the disease — that is **overfitting**, and the figure below shows it as the orange line rising while the blue line falls.

Change `MODEL` (and optionally `VALUES`) and re-run.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Pick a model, then re-run. Try them all:
#     'logistic'          -> C: smaller = simpler, more regularised
#     'knn'               -> n_neighbors: 1 memorises, 51 over-smooths
#     'tree'              -> max_depth: how many questions deep
#     'random_forest'     -> max_depth
#     'svm'               -> C: how hard it tries to get every point right
#     'gradient_boosting' -> learning_rate
# ==========================================================================
MODEL = 'random_forest'

description, parameter, values = MODEL_CHOICES[MODEL]
print(f'{MODEL}: {description}')
print(f'Sweeping {parameter} over {values}\n')

swept, train_scores, test_scores = sweep_parameter(
    MODEL, parameter, values, X_train, y_train, X_test, y_test)

plots.plot_parameter_sweep(swept, train_scores, test_scores, parameter,
                           title=f'{MODEL}: the gap between the lines IS the overfitting')
plt.show()
best = values[int(np.argmax(test_scores))]
print(f'Best held-out {parameter} for {MODEL}: {best}')


🧠 **Think first:** Why is it cheating to pick the setting with the best *held-out* score and then report that score as your result?

<details>
<summary>Click for one good answer</summary>

Because you just used the test set to make a decision, so it is no longer held out — you have leaked, gently, through your own choices. The honest procedure is three-way: train on one part, tune on a *validation* part, and only ever touch the test part once, at the very end. With 420 people that is wasteful, which is why cross-validation (🔵 below) is the usual answer in biomedicine.

</details>


### 3.4 🔵 Your turn to write code — cross-validation

A single train/test split of 420 people is noisy: shuffle differently and the AUROC moves by several points. **Cross-validation** splits the data five ways, trains five times, and reports the spread — far more trustworthy at biomedical sample sizes.

The helper is already imported for you. Fill in the `# TODO` line.

> **How practitioners think about this:** at n = 300–500, the error bar on your score is often bigger than the difference between two models. Before you claim model A beats model B, check whether their cross-validation ranges overlap. If they do, you have not shown anything.


In [ ]:
from models import cross_validated_score

# ✅ Worked solution.
fold_scores = {}
fold_scores['logistic'] = cross_validated_score('logistic', X, y, folds=5, metric='auroc')
fold_scores['random_forest'] = cross_validated_score('random_forest', X, y, folds=5, metric='auroc')

# Why this matters: notice we pass the FULL X and y, not X_train. Cross-validation makes
# its own five splits internally, and the preprocessing is refitted inside every fold, so
# nothing leaks. If we had passed pre-scaled data, we would have reintroduced the exact
# leak we removed in section 2.5.
#
# Read the output as a range, not a number. If logistic gives 0.91 ± 0.03 and the forest
# gives 0.89 ± 0.04, those are the same result. Reporting 'logistic won' would be noise.

fig, ax = plt.subplots(figsize=(6, 3.8))
for position, (name, scores) in enumerate(fold_scores.items()):
    ax.scatter([position] * len(scores), scores, s=60, color='#2c6fbb', zorder=3)
    ax.plot([position - 0.18, position + 0.18], [np.mean(scores)] * 2, color='#e08214', linewidth=3)
    ax.annotate(f'{np.mean(scores):.3f}\n± {np.std(scores):.3f}', (position, np.mean(scores)),
                textcoords='offset points', xytext=(24, -6), fontsize=9)
ax.set_xticks(range(len(fold_scores)), list(fold_scores), fontsize=10)
ax.set_ylabel('AUROC'); ax.set_ylim(0.5, 1.02)
ax.set_title('Five folds each. Do the clouds overlap?')
plt.tight_layout(); plt.show()


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** run 3.3 for `'knn'` with `n_neighbors = 1` and explain, in one sentence, why the orange training line hits 1.0.
- 🔵 **If you want to write code:** add `'gradient_boosting'` to the ladder in 3.2 and see whether boosting beats the forest here. Then try giving the forest `n_estimators=1` — how much of a forest's power is the averaging?
- ⚫ **Take home:** implement a nested cross-validation: an inner loop that picks the hyperparameter and an outer loop that scores it. This is the only fully honest way to report a tuned model's performance.


---
# 4 · Read the results

A number is not a result. In this section every cell produces a **figure**, because the questions that matter — *who does this model fail? what does a mistake cost? can I trust the probability?* — are not answerable from a single score.


### 4.1 The four standard views

We fit one model — logistic regression on the blood markers, no MMSE — and look at it four ways.


In [ ]:
final_model = train_model('logistic', X_train, y_train)
probability = final_model.predict_proba(X_test)[:, 1]
predicted = (probability >= 0.5).astype(int)
final_metrics = evaluate(final_model, X_test, y_test)

plots.plot_confusion(y_test, predicted, labels=('not AD', 'AD'),
                     title='Held-out patients: what the model got right and wrong')
plt.show()

plots.plot_roc_pr(y_test, probability, title='Blood-biomarker model, held-out patients')
plt.show()

plots.plot_calibration(y_test, probability)
plt.show()

for name, value in final_metrics.items():
    print(f'  {name:<20s} {value:.3f}')


**How to read these.**

- **Confusion matrix** — the bottom-left box is a *missed diagnosis*. In early AD that may mean a person is not offered an anti-amyloid therapy while it could still help, and is not told what is happening to them. The top-right box is a *false alarm*: an unnecessary lumbar puncture or PET scan, cost, and months of fear. These are not interchangeable, and no single accuracy number can tell them apart.
- **ROC** — performance across every possible cutoff. AUROC of 0.5 is a coin flip, 1.0 is perfect.
- **Precision-recall** — the same model, but it *notices* when cases are rare. The dashed line is what you would get by guessing at random. Under imbalance this is the more honest curve.
- **Calibration** — when the model says "70% chance", do 70% of those people actually have AD? A model can rank patients perfectly (great AUROC) and still be badly calibrated, which makes its probabilities useless for a conversation with a patient.


### 4.2 ✏️ Your turn — move the threshold, change the medicine

The model outputs a probability. Turning that into a yes/no needs a **threshold**, and that choice is a *clinical* decision, not a statistical one.

- A **screening** test wants a low threshold: catch everyone, tolerate false alarms.
- A **confirmatory** test wants a high threshold: be sure before you act.

Change `DECISION_THRESHOLD` and watch both the curve and the confusion matrix move.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Try 0.20, then 0.50, then 0.80.
#   For each: how many AD patients did you miss, and how many
#   healthy people did you send for an unnecessary lumbar puncture?
# ==========================================================================
DECISION_THRESHOLD = 0.5

plots.plot_threshold_sweep(y_test, probability, chosen=DECISION_THRESHOLD)
plt.show()

at_threshold = (probability >= DECISION_THRESHOLD).astype(int)
plots.plot_confusion(y_test, at_threshold, labels=('not AD', 'AD'),
                     title=f'Decisions at threshold {DECISION_THRESHOLD}')
plt.show()

shifted = evaluate(final_model, X_test, y_test, threshold=DECISION_THRESHOLD)
print(f"At {DECISION_THRESHOLD}: sensitivity {shifted['sensitivity']:.2f} "
      f"(share of AD patients caught), specificity {shifted['specificity']:.2f} "
      f"(share of healthy people spared).")
print(f'You would send {at_threshold.mean():.0%} of this clinic for further testing.')


### 4.3 What is the model actually using? Shapley values

Coefficients tell you about the model. **Shapley values** tell you about *each individual patient*: how much did this person's high p-tau, specifically, push their predicted risk up?

The idea comes from game theory. Treat the features as players in a team, and the prediction as the prize. A feature's Shapley value is its fair share of the prize, averaged over every possible order in which the team could have been assembled. We compute the **exact** value here by enumerating all 2⁸ = 256 combinations — the well-known `shap` package approximates this because real models have hundreds of features, but with eight we do not need to.


In [ ]:
from interpret import shapley_values, shapley_importance, baseline_prediction

explain_columns = ['age', 'ab42_40_ratio', 'ptau181_pg_ml', 'nfl_pg_ml',
                   'gfap_pg_ml', 'apoe4_carrier', 'education_years']
shap_frame = shapley_values(final_model, X_test.head(60), X_train, features=explain_columns)

# (a) Globally: which features move predictions the most, across 60 held-out patients?
importance = shapley_importance(shap_frame)
plots.plot_importance(importance.index, importance.values,
                      title='Average influence on predicted AD risk (exact Shapley values)',
                      xlabel='mean |contribution| to predicted probability')
plt.show()

# (b) For ONE patient: why did the model say what it said?
PATIENT = 0
one = shap_frame.iloc[PATIENT].sort_values()
plots.plot_importance(one.index, one.values,
                      title=f'Patient {X_test.index[PATIENT]}: what pushed this prediction up (blue) and down (red)',
                      xlabel='contribution to predicted probability')
plt.show()

base = baseline_prediction(final_model, X_train)
print(f'Average predicted risk across the cohort: {base:.3f}')
print(f'Sum of this patient\'s contributions:      {one.sum():+.3f}')
print(f'Model\'s prediction for this patient:      {base + one.sum():.3f}')
print(f'(Check — the model really predicts:        {probability[PATIENT]:.3f})')
print('\nThose two numbers agreeing is not a coincidence: Shapley values are defined to add up.')


🧠 **Think first:** The Shapley plot says `age` is influential. Is the model detecting Alzheimer's disease, or detecting old age?

<details>
<summary>Click for one good answer</summary>

Both, unavoidably entangled — and you cannot tell which from this figure alone. Age is the strongest risk factor for AD, so an age-heavy model will score well while adding nothing a calendar could not. The test is to remove age and see what survives: if the biomarkers still work, they carry independent information. **Module D is entirely about this problem** — it is a good choice for your second module.

</details>


### 4.4 Who does this model fail?

An overall score hides the people it was worst for. Split the errors by subgroup and look.


In [ ]:
errors = df.loc[X_test.index].copy()
errors['correct'] = (predicted == y_test).astype(int)
errors['age_band'] = pd.cut(errors['age'], [50, 65, 75, 85, 100],
                            labels=['50-65', '65-75', '75-85', '85+'])

for subgroup in ['sex', 'site', 'age_band', 'diagnosis']:
    plots.plot_subgroup_errors(errors, subgroup, 'correct',
                               title=f'Proportion correct by {subgroup}')
    plt.show()


**The MCI bar is the one to look at.** People with mild cognitive impairment are counted as 'not AD' here, but many of them will convert to AD within a few years. The model is being marked wrong for flagging people who are arguably early cases — and marked right for reassuring people who are about to get worse. A label is a snapshot; the disease is a process.

**And the bias question.** This simulated cohort is deliberately narrow: three clinics, one broad ancestry group, everyone already referred to a memory service. A model trained here has never seen the people least likely to be referred in the first place — which in most health systems means poorer, less educated, and minority-ethnic patients. Deploying it would work worst for exactly the groups already worst served.


### 4.5 Your headline result

One figure summarising what you built, for your own notes. **There is no shared scoreboard and no comparison between students** — the interesting differences between modules are qualitative, and we discuss them together at the end.


In [ ]:
summary = pd.Series(final_metrics)
plots.plot_score_comparison(list(summary.index), list(summary.values), reference=0.5,
                            colours=['#2c6fbb'] * len(summary),
                            title='Module C — blood biomarkers, logistic regression, held-out patients',
                            ylabel='score')
plt.show()

print('Model:      logistic regression on 8 blood/CSF and demographic features')
print(f'Trained on: {len(X_train)} people   Tested on: {len(X_test)} unseen people')
print('Excluded:   MMSE (label leakage), site (batch confounder)')
for name, value in final_metrics.items():
    print(f'  {name:<20s} {value:.3f}')


### 4.6 What would have to be true before this touched a patient?

Honest limitations, in the order a regulator would ask about them:

1. **The data are simulated.** Nothing here is evidence about real biomarkers. Real p-tau217 assays do perform roughly this well against clinical diagnosis — but *roughly this well in published cohorts* is not the same as *this well in your clinic*.
2. **One cohort, one time.** The model has never seen a different hospital, a different assay lot, or a different population. External validation is not a nice-to-have; it is the whole test.
3. **The label is imperfect.** Clinical AD diagnosis without autopsy or PET confirmation is wrong perhaps 10–30% of the time. The model can never be more accurate than its labels.
4. **Nobody has shown it changes anything.** A model that predicts well but does not alter treatment, timing or outcome has no clinical value. That requires a prospective study, not a held-out split.
5. **Coverage.** See 4.4. If the training cohort excluded a group, the model's confident answers about that group are confident guesses.

---

### 🧠 Final question for the group discussion

Blood tests are cheap, quick and nearly harmless. Lumbar punctures and PET scans are expensive, unpleasant and rationed. Given the confusion matrix you produced in 4.2 — **would you use this model to decide who gets referred for a PET scan?** What threshold would you set, and who would you be willing to miss?


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** set `DECISION_THRESHOLD` in 4.2 to whatever you would defend in a clinic, and be ready to say why.
- 🔵 **If you want to write code:** in 4.3, change `PATIENT` to find a patient the model got *wrong*, and use their Shapley plot to explain what misled it. (Hint: `wrong = np.where(predicted != y_test)[0]`.)
- ⚫ **Take home:** the `shap` package draws beeswarm and waterfall plots from these same values. Install it and compare its `KernelExplainer` output to our exact values — how close does the approximation get?
